# 第 09 课｜线性回归与最小二乘

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`09_youtube-O5IPAeu8X6w/index.html`  
> **所属模块**：P1-M3 · 统计证据与可预测性

## 本节量化问题

策略收益有多少来自市场暴露，有多少是回归残差或截距？

## 准确定义与策略报告位置

OLS 用最小残差平方估计线性关系；Beta 是市场暴露，Alpha 是模型条件下的截距。

- **核心关系**：R_strategy=α+βR_market+ε；β=Cov(R_strategy,R_market)/Var(R_market)。
- **写入报告**：基准归因、Beta 和 Alpha
- **P2 衔接**：P2 第 49 课：statsmodels 回归与稳健推断。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 用市场收益解释策略收益

`add_constant` 增加截距 α；Beta 是市场收益系数。稳健标准误只修正推断，不改变 OLS 点估计。

In [2]:
import statsmodels.api as sm

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
returns = close.pct_change(fill_method=None).dropna()

y = returns["SIM_GROWTH"]
X = sm.add_constant(returns[["SIM_MARKET"]])
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

result = pd.Series({
    "alpha_daily": model.params["const"],
    "beta_market": model.params["SIM_MARKET"],
    "r_squared": model.rsquared,
    "alpha_pvalue_hac": model.pvalues["const"],
})
display(result)

alpha_daily         0.000068
beta_market         1.203063
r_squared           0.563383
alpha_pvalue_hac    0.836163
dtype: float64

## 方法边界与阶段验收

Alpha 依赖基准和因子设定；异方差、自相关和遗漏变量会破坏普通标准误。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
